# 2. Train the baseline

Run `prepare_data.ipynb` first. In Colab, turn on a GPU: **Runtime > Change runtime type > GPU**.

This first run is your **baseline**, the single set of hyperparameters the brief asks for. Later experiments get compared against it, so keep its results.

In [ ]:
!pip install -q ultralytics

In [ ]:
import shutil
from pathlib import Path

from ultralytics import YOLO

ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
DATA_YAML = ROOT / 'datasets' / 'merged' / 'data.yaml'
RUNS_DIR = ROOT / 'runs'

try:
    from google.colab import drive
except ImportError:
    pass
else:
    # Load the dataset from Drive, and save results there so a disconnect can't lose them.
    drive.mount('/content/drive')
    drive_dir = Path('/content/drive/MyDrive/minecraft-ore-yolo')
    RUNS_DIR = drive_dir / 'runs'
    if not DATA_YAML.exists():
        shutil.unpack_archive(drive_dir / 'merged.zip', DATA_YAML.parent)

assert DATA_YAML.exists(), f'{DATA_YAML} not found. Run prepare_data.ipynb first.'

## Quick test

Two epochs on a small slice of the data, just to catch setup problems before the long run. Ignore the scores.

In [ ]:
YOLO('yolov8n.pt').train(data=str(DATA_YAML), epochs=2, imgsz=640, fraction=0.05, project=str(RUNS_DIR), name='smoke_test')

## Baseline

`yolov8n.pt` starts from COCO-pretrained weights, so this is transfer learning. Write these settings down for your report.

In [ ]:
BASELINE = dict(epochs=50, imgsz=640, batch=16, seed=42)

model = YOLO('yolov8n.pt')
model.train(data=str(DATA_YAML), project=str(RUNS_DIR), name='baseline', **BASELINE)

## Evaluate

Use the **validation** split while making choices. Save the **test** split for your final numbers, so they stay an honest measure.

- **mAP50:** detection score, counting a box as correct if it overlaps the true box by at least 50%.
- **mAP50-95:** the same, but averaged over stricter overlap requirements.

In [ ]:
best_weights = model.trainer.best  # or RUNS_DIR / 'baseline' / 'weights' / 'best.pt' in a new session
best = YOLO(best_weights)

metrics = best.val(data=str(DATA_YAML), split='val')
print(f'mAP50: {metrics.box.map50:.3f}')
print(f'mAP50-95: {metrics.box.map:.3f}')
for i, class_map in zip(metrics.box.ap_class_index, metrics.box.maps[metrics.box.ap_class_index]):
    print(f'  {best.names[int(i)]:<15} {class_map:.3f}')

## Try it on an image

Set `IMAGE` to one of your own screenshots. `save=True` writes the annotated copy under `runs/`.

In [ ]:
IMAGE = 'path/to/your/screenshot.png'  # TODO

for result in best(IMAGE, save=True):
    for box in result.boxes:
        print(best.names[int(box.cls)], f'{float(box.conf):.2f}')